# Medallion Architecture — Notebook 2: Saving Data to the Bronze Layer

This is the second notebook in the Medallion Architecture tutorial series. In Notebook 1 we read `Products.csv` into a Spark DataFrame and explored it. Here we write that raw data **as-is** into a Delta table — the **Bronze** layer of the medallion architecture.

## The Bronze Layer

In the medallion architecture, data flows through three layers: **Bronze → Silver → Gold**.

The **Bronze** layer is the first stop: raw data landed **as-is**, in its original structure. We don't clean, deduplicate, or reshape anything here — Bronze just makes the raw data reliably available as a table.

In this notebook we:

1. Re-create the Spark DataFrame from `Data/Products.csv` (so this notebook runs standalone)
2. Write it as the Delta table `demo.demo_sch.products_bronze`
3. Verify the write by reading the table back

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType,
)

# Same explicit schema as Notebook 1 — repeated so this notebook runs standalone.
products_schema = StructType([
    StructField("OrderDate",          DateType(),    False),
    StructField("InvoiceNum",         StringType(),  False),
    StructField("ProductKey",         IntegerType(), False),
    StructField("Country",            StringType(),  False),
    StructField("EnglishProductName", StringType(), False),
    StructField("SalesAmount",        DoubleType(),  False),
    StructField("UnitPrice",          DoubleType(),  False),
    StructField("OrderQuantity",      IntegerType(), False),
    StructField("TaxAmt",             DoubleType(),  False),
    StructField("TotalProductCost",   DoubleType(),  False),
])

csv_path = "/Workspace/Users/surabhid5@gmail.com/Databricks-Medallion-Architecture/Data/Products.csv"

products_df = (
    spark.read
    .option("header", True)
    .option("dateFormat", "dd-MM-yyyy")
    .schema(products_schema)
    .csv(f"file:{csv_path}")
)

row_count = products_df.count()
print(f"Loaded {row_count:,} rows from Products.csv")

## Save as a Delta Table (Bronze Layer)

The Bronze layer holds raw, ingested data as-is. We write our Spark DataFrame to the Unity Catalog table `demo.demo_sch.products_bronze` in **Delta** format, using `mode("overwrite")` so this notebook stays safely re-runnable.

Delta gives us ACID transactions, schema enforcement, and time travel — the foundation the Silver and Gold layers build on.

In [0]:
bronze_table = "demo.demo_sch.products_bronze"

(
    products_df.write
    .format("delta")
    .mode("overwrite")          # makes the notebook safely re-runnable
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print(f"Wrote {row_count:,} rows to {bronze_table}")

In [0]:
# Read the table back from the metastore and confirm it matches the source
bronze_df = spark.table(bronze_table)

print(f"Rows in {bronze_table}: {bronze_df.count():,}")
print(f"Row counts match source: {bronze_df.count() == row_count}")

bronze_df.printSchema()
display(bronze_df.limit(5))

## Recap

* Re-created the raw sales DataFrame from `Data/Products.csv` — raw data, unchanged
* Wrote it as a Delta table — the **Bronze layer**: `demo.demo_sch.products_bronze`
* Verified the write by reading the table back and comparing row counts

**Next up:** Notebook 3 — clean and transform the Bronze table into the **Silver** layer.